# 8.8 — LSTM

An LSTM separates memory from exposure, using gates to decide what to keep, what to write, and what to reveal.

This lesson is the classic answer to the vanilla RNN's fading-memory problem. LSTM adds a dedicated cell state beside the hidden state, so memory preservation can be a multiplication by a forget gate rather than a full nonlinear rewrite at every step.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np

SEED = 87
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def softmax(scores, axis=-1):
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=axis, keepdims=True)


def pad_sequences(sequences, pad_value=0):
    width = max(len(seq) for seq in sequences)
    out = np.full((len(sequences), width), pad_value, dtype=int)
    mask = np.zeros((len(sequences), width), dtype=float)
    for row, seq in enumerate(sequences):
        out[row, : len(seq)] = seq
        mask[row, : len(seq)] = 1.0
    return out, mask


def summarize_rung(rung):
    lengths = [len(seq) for seq in rung["sequences"]]
    vocab = sorted({token for seq in rung["sequences"] for token in seq})
    return {
        "name": rung["name"],
        "n": len(rung["sequences"]),
        "min_len": min(lengths),
        "max_len": max(lengths),
        "vocab": vocab,
    }


def build_sequence_classification_ladder(kind):
    base = [
        {"name": "D1 lesson toy", "sequences": [[1, 0, 1, 1]], "labels": [1], "dependency": 4},
        {"name": "D2 clean patterns", "sequences": [[1, 0, 1], [0, 0, 1], [1, 1, 0], [0, 1, 0], [1, 0, 0]], "labels": [1, 0, 1, 0, 0], "dependency": 3},
        {"name": "D3 long gaps and noise", "sequences": [[1, 0, 0, 1], [1, 2, 0, 0], [0, 2, 2, 1], [0, 0, 0, 0], [1, 0, 2, 1], [0, 1, 2, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 4},
        {"name": "D4 click/dialogue snippets", "sequences": [[3, 1, 0, 4], [2, 0, 4], [3, 2, 2, 4], [1, 1, 0, 0], [3, 0, 1, 4], [2, 2, 0, 4]], "labels": [1, 0, 1, 0, 1, 0], "dependency": 5},
        {"name": "D5 delayed dependency", "sequences": [[1, 0, 0, 0, 0, 4], [0, 0, 1, 0, 0, 4], [1, 2, 2, 2, 2, 0], [0, 2, 2, 2, 2, 4], [1, 0, 2, 0, 2, 4], [0, 0, 0, 0, 0, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 6},
    ]
    if kind == "lstm":
        base[0] = {"name": "D1 lesson gate toy", "sequences": [[2, 0, 2]], "labels": [1], "dependency": 3}
    if kind == "gru":
        base[0] = {"name": "D1 lesson interpolation toy", "sequences": [[2, 1, 0, 1]], "labels": [1], "dependency": 4}
    return base


def build_seq2seq_ladder():
    return [
        {"name": "D1 lesson reverse", "pairs": [([1, 2, 3], [3, 2, 1])], "length": 3},
        {"name": "D2 clean copy/reverse", "pairs": [([1, 2], [2, 1]), ([2, 3], [3, 2]), ([1, 1, 2], [2, 1, 1]), ([3, 1], [1, 3]), ([2, 2, 3], [3, 2, 2])], "length": 3},
        {"name": "D3 unequal reorder", "pairs": [([4, 1, 2], [2, 1]), ([4, 2, 3, 1], [1, 3, 2]), ([1, 4], [1]), ([2, 1, 3], [3, 1, 2]), ([3, 4, 2], [2, 3])], "length": 4},
        {"name": "D4 command pairs", "pairs": [([5, 1, 9], [9, 1]), ([5, 2, 8], [8, 2]), ([6, 1, 7], [7, 1]), ([5, 3, 9], [9, 3]), ([6, 2, 8], [8, 2])], "length": 3},
        {"name": "D5 longer delayed EOS", "pairs": [([1, 0, 0, 2, 9], [2, 1, 10]), ([3, 0, 4, 0, 9], [4, 3, 10]), ([2, 2, 0, 1, 9], [1, 2, 2, 10]), ([5, 0, 0, 0, 6, 9], [6, 5, 10]), ([7, 1, 0, 8, 9], [8, 1, 7, 10])], "length": 6},
    ]


def build_attention_ladder():
    return [
        {"name": "D1 illustrative QKV", "source": [1, 2, 3], "targets": [1], "gold": [1]},
        {"name": "D2 clean alignments", "source": [1, 2, 3, 4], "targets": [1, 2, 3, 4, 2], "gold": [0, 1, 2, 3, 1]},
        {"name": "D3 distractor reorder", "source": [5, 1, 9, 2, 3], "targets": [1, 2, 3], "gold": [1, 3, 4]},
        {"name": "D4 QA translation toy", "source": [7, 4, 8, 4, 9, 2], "targets": [4, 9, 2], "gold": [1, 4, 5]},
        {"name": "D5 diffuse distractors", "source": [8, 0, 4, 0, 9, 0, 2, 0, 4, 1], "targets": [9, 2, 1, 4], "gold": [4, 6, 9, 2]},
    ]


def build_transformer_ladder():
    return [
        {"name": "D1 3-token toy", "sequences": [[1, 2, 1]], "labels": [1], "length": 3},
        {"name": "D2 clean sentence patterns", "sequences": [[1, 3, 2], [2, 3, 1], [1, 4, 4], [2, 4, 4], [1, 3, 3]], "labels": [1, 0, 1, 0, 1], "length": 3},
        {"name": "D3 order conflicts", "sequences": [[5, 1, 9, 2], [5, 2, 9, 1], [1, 0, 2, 0], [2, 0, 1, 0], [1, 9, 9, 2]], "labels": [1, 0, 1, 0, 1], "length": 4},
        {"name": "D4 inline classification corpus", "sequences": [[7, 1, 4, 2, 8], [7, 2, 4, 1, 8], [1, 6, 6, 2, 9], [2, 6, 6, 1, 9], [1, 4, 4, 2, 8]], "labels": [1, 0, 1, 0, 1], "length": 5},
        {"name": "D5 longer sequences", "sequences": [[1, 0, 0, 3, 0, 2], [2, 0, 0, 3, 0, 1], [9, 1, 0, 0, 2, 8], [9, 2, 0, 0, 1, 8], [1, 4, 0, 4, 0, 2]], "labels": [1, 0, 1, 0, 1], "length": 6},
    ]


def accuracy(predictions, labels):
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    return float(np.mean(predictions == labels))


## The concept, built once (D1)

An LSTM keeps a cell state beside the hidden state. The core memory line is $$c_t=f_t\odot c_{t-1}+i_t\odot\tilde c_t$$, so old storage and new writes are visible terms.

In [ ]:

def lstm_step(x_t, h_prev, c_prev, gates):
    f_t = np.asarray(gates["f"], dtype=float)
    i_t = np.asarray(gates["i"], dtype=float)
    o_t = np.asarray(gates["o"], dtype=float)
    g_t = np.tanh(np.asarray(gates["g"], dtype=float))
    c_t = f_t * c_prev + i_t * g_t
    h_t = o_t * np.tanh(c_t)
    return h_t, c_t

kept_values = []
for forget in [0.0, 0.5, 1.0]:
    _, c_next = lstm_step(0.0, 0.0, 2.0, {"f": forget, "i": 0.0, "o": 1.0, "g": 0.0})
    kept_values.append(float(c_next))
assert np.allclose(kept_values, [0.0, 1.0, 2.0])
kept_values


The lesson also stresses that the candidate write is nonlinear: $\tilde c_t=\tanh(2.0)=0.9640275800758169$, not raw 2.0. The hidden state can be small while the cell is substantial because the output gate controls exposure.

In [ ]:

g_candidate = math.tanh(2.0)
assert np.isclose(g_candidate, 0.9640275800758169)
h_next, c_next = lstm_step(0.0, 0.0, 1.0, {"f": 0.8, "i": 0.5, "o": 0.8, "g": 2.0})
assert np.isclose(c_next, 1.2820137900379085)
assert np.isclose(math.tanh(c_next), 0.8570205322353923)
assert np.isclose(0.2 * math.tanh(c_next), 0.17140410644707849)
print("cell", c_next, "hidden with o=0.8", h_next)


## The dataset ladder

The D1--D5 ladder uses synthetic memory sequences of rising delay and distractor structure.

In [ ]:

rungs = build_sequence_classification_ladder("lstm")
for rung in rungs:
    print(summarize_rung(rung))
    print("sample", rung["sequences"][0], "label", rung["labels"][0])


## Run the same method across D1--D5

The classifier uses a real LSTM update with hand-set gates: important tokens write, distractors mostly keep, and output exposure remains separate from storage.

In [ ]:

def lstm_sequence_classifier(seq, forget_bias=2.4):
    h = 0.0
    c = 0.0
    cells = []
    gates_seen = []
    for token in seq:
        informative = 1.0 if token in [1, 2, 3, 4] else 0.0
        positive = 1.0 if token in [1, 3, 4] else -1.0
        f = sigmoid(forget_bias)
        i = sigmoid(2.0 * informative - 1.0)
        o = sigmoid(1.0)
        g = positive
        h, c = lstm_step(float(token), h, c, {"f": f, "i": i, "o": o, "g": g})
        cells.append(float(c))
        gates_seen.append([float(f), float(i), float(o)])
    prediction = int(c > 0.35)
    return prediction, float(c), np.asarray(cells), np.asarray(gates_seen)

results = []
for rung in rungs:
    preds = []
    for seq in rung["sequences"]:
        pred, score, cells, gates_seen = lstm_sequence_classifier(seq)
        preds.append(pred)
    results.append({"rung": rung["name"], "delay": rung["dependency"], "accuracy": accuracy(preds, rung["labels"])})

for row in results:
    print(f'{row["rung"]:30s} delay={row["delay"]:2d} accuracy={row["accuracy"]:.3f}')


## Results visualization

The panels show cell-state memory on sample sequences, followed by accuracy against delay.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for index, rung in enumerate(rungs):
    _, _, cells, gates_seen = lstm_sequence_classifier(rung["sequences"][0])
    heat = np.vstack([cells, gates_seen[:, 0], gates_seen[:, 1], gates_seen[:, 2]])
    axes[0, index].imshow(heat, aspect="auto", cmap="magma")
    axes[0, index].set_title(rung["name"].split()[0])
    axes[0, index].set_yticks([0, 1, 2, 3])
    axes[0, index].set_yticklabels(["c", "f", "i", "o"])

axes[1, 0].plot([row["delay"] for row in results], [row["accuracy"] for row in results], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_xlabel("delay")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_title("metric curve")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on D5: hidden state is not the memory cell

A low forget gate destroys long-range information. Plotting only $h_t$ can also hide the fact that $c_t$ stores more than the exposed output.

In [ ]:

retained = 0.95 ** 30
assert np.isclose(retained, 0.21463876394293727)
d5 = rungs[-1]
low_forget_preds = [lstm_sequence_classifier(seq, forget_bias=-2.0)[0] for seq in d5["sequences"]]
keep_preds = [lstm_sequence_classifier(seq, forget_bias=2.4)[0] for seq in d5["sequences"]]
print("wrong low-forget D5 accuracy", accuracy(low_forget_preds, d5["labels"]))
print("fixed keep-gate D5 accuracy", accuracy(keep_preds, d5["labels"]))
print("cell retained after 30 keep steps", retained)


## Evaluate it + Practice

- Metric: sequence classification accuracy.
- No-skill baseline: majority class on the rung.
- Cheap sanity check: forget gates [0, 0.5, 1] keep [0, 1, 2].
- Ablation: force the forget bias low and inspect D5.
- Failure signals: cell heatmap keeps evidence while hidden output is small.

Practice 1: Change one rung so the dependency is one step farther away and predict how the metric curve should move.

Practice 2: Change the output gate and verify that $c_t$ need not change.

Practice 3: Add a distractor write and lower only the input gate.